In [1]:
!pip install -q gradio

In [2]:
from google.colab import files
uploaded = files.upload()  # select your traffic_sign_model.h5 file here

Saving traffic_sign_model.h5 to traffic_sign_model.h5


In [3]:
import tensorflow as tf
import numpy as np
import cv2
import gradio as gr

model = tf.keras.models.load_model("traffic_sign_model.h5")
IMG_SIZE = 32

# GTSRB class labels (ClassId -> sign name)
class_names = {
    0: 'Speed limit (20km/h)', 1: 'Speed limit (30km/h)', 2: 'Speed limit (50km/h)',
    3: 'Speed limit (60km/h)', 4: 'Speed limit (70km/h)', 5: 'Speed limit (80km/h)',
    6: 'End of speed limit (80km/h)', 7: 'Speed limit (100km/h)', 8: 'Speed limit (120km/h)',
    9: 'No passing', 10: 'No passing veh over 3.5 tons', 11: 'Right-of-way at intersection',
    12: 'Priority road', 13: 'Yield', 14: 'Stop', 15: 'No vehicles',
    16: 'Veh > 3.5 tons prohibited', 17: 'No entry', 18: 'General caution',
    19: 'Dangerous curve left', 20: 'Dangerous curve right', 21: 'Double curve',
    22: 'Bumpy road', 23: 'Slippery road', 24: 'Road narrows on the right',
    25: 'Road work', 26: 'Traffic signals', 27: 'Pedestrians', 28: 'Children crossing',
    29: 'Bicycles crossing', 30: 'Beware of ice/snow', 31: 'Wild animals crossing',
    32: 'End speed + passing limits', 33: 'Turn right ahead', 34: 'Turn left ahead',
    35: 'Ahead only', 36: 'Go straight or right', 37: 'Go straight or left',
    38: 'Keep right', 39: 'Keep left', 40: 'Roundabout mandatory',
    41: 'End of no passing', 42: 'End no passing veh > 3.5 tons'
}

In [4]:
def predict_sign(image):
    if image is None:
        return "No image provided"

    img = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    predictions = model.predict(img)[0]
    top3_idx = predictions.argsort()[-3:][::-1]

    result = {class_names[i]: float(predictions[i]) for i in top3_idx}
    return result

In [5]:
demo = gr.Interface(
    fn=predict_sign,
    inputs=gr.Image(type="numpy", label="Upload a Traffic Sign Image"),
    outputs=gr.Label(num_top_classes=3, label="Prediction"),
    title="🚦 Traffic Sign Recognition",
    description="Upload an image of a traffic sign and the model will predict its class.",
    examples=None
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8e910e2480207357db.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
